# SRM reconstruction + renderer-guided flow refinement

A compact Colab workflow for the two parts of this project:

1. Reconstruct the initial specular reflectance maps (SRMs) using the paper's Section 3.2 objective.
2. Refine those SRMs with the experimental scene-specific rectified-flow prior and captured-frame guidance.

Use a GPU runtime: **Runtime -> Change runtime type -> T4 GPU**. The flow stage is an add-on, not part of the original paper. Also, an SRM is the environment convolved with the object's specular BRDF, so a rough surface is intrinsically blurry; the flow stage cannot recover detail that was never observed.

## 1. Put the project on Google Drive

Upload the complete `cs180_final_project` folder to `MyDrive`. It must contain the Python files plus either `cache/` and `cans_data/`, or a prepared synthetic scene under `scenes/<name>/`. Edit `DRIVE_PROJECT` if your folder is elsewhere.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/cs180_final_project'
WORK = '/content/cs180_final_project'
SCENE = ''  # '' = real Cans data; e.g. 'two' = scenes/two

# Copy to Colab's local disk: training directly on Drive is much slower.
!mkdir -p "$WORK"
!rsync -a --exclude='.venv' --exclude='__pycache__' "$DRIVE_PROJECT/" "$WORK/"
%cd $WORK
!pip -q install numpy opencv-python torch torchvision matplotlib
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Validate the prepared inputs before spending GPU time.
from pathlib import Path
import os, subprocess, shlex

base = Path(WORK) / ('scenes/' + SCENE if SCENE else '')
required = [base/'cache/rgb.npy', base/'cache/diffuse.npy', base/'cache/refl.npy']
# refl.npy is computed on demand by data.py; the actual required geometry buffers are below.
required = [base/'cache/rgb.npy', base/'cache/diffuse.npy', base/'cache/depth.npy',
            base/'cache/normal.npy', base/'cache/vis.npy', base/'cache/split.npz']
missing = [str(p) for p in required if not p.exists()]
assert not missing, 'Missing prepared inputs:\n' + '\n'.join(missing)

env = os.environ.copy()
if SCENE:
    env['SCENE'] = SCENE

def run(command):
    print('$', command, flush=True)
    subprocess.run(shlex.split(command), cwd=WORK, env=env, check=True)

print('Inputs found in', base)

## 2. Train the paper-style initial SRM

This optimizes the panorama pixels, material-weight network, and diffuse-refinement network. Perceptual and GAN losses are enabled, matching the paper's reported objective more closely than the fast CPU test. `--cans both` is appropriate for the supplied two-can data; change it to `a`, `b`, or `all` for another capture.

Start with 8,000 steps. If the panorama is still underfit, resume with a larger budget.

In [ ]:
SRM_RUN = 'srm_colab'
run(f'python train.py srm --run {SRM_RUN} --cans both --materials 2 '
    '--max-steps 8000 --lr-srm 0.001 --lambda-p 1.0 --lambda-g 0.05 '
    '--batch 2 --log 100 --preview 1000')

In [ ]:
# Evaluate and inspect the initial reconstruction.
from IPython.display import Image, display
run(f'python eval_env.py --run {SRM_RUN} --max-frames 128')
initial_result = base / 'results' / SRM_RUN / 'pano.png'
display(Image(filename=str(initial_result)))

## 3. Renderer-guided rectified-flow refinement

The flow is trained around the reconstructed SRM; it does not use a reference panorama. Small noise and guidance values avoid replacing learned structure with colored noise. The captured RGB frames provide the guidance through the differentiable specular renderer.

In [ ]:
FLOW_RUN = 'srm_colab_flow'
run(f'python flow_refine.py --srm-run {SRM_RUN} --run {FLOW_RUN} '
    '--prior-steps 500 --ode-steps 30 --frame-batch 4 '
    '--sigma 0.02 --guidance 0.005')
run(f'python eval_env.py --run {FLOW_RUN} --max-frames 128')

In [ ]:
# Compare initial and flow-refined panoramas.
print('Initial SRM')
display(Image(filename=str(base/'results'/SRM_RUN/'pano.png')))
print('Flow-refined SRM')
display(Image(filename=str(base/'results'/FLOW_RUN/'pano.png')))

## 4. Optional: paper's final neural renderer

This is needed for novel-view RGB synthesis, not for viewing the recovered environment panorama. It trains the paper's Section 3.3 renderer from diffuse, specular, first-bounce, reflection-direction, and Fresnel buffers.

In [ ]:
# Uncomment to train the complete paper renderer.
# run(f'python train.py renderer --srm-run {FLOW_RUN} --run renderer_colab --batch 2')
# run(f'python evaluate.py --srm-run {FLOW_RUN} --renderer-run renderer_colab')

## 5. Save outputs back to Drive

Colab's local disk is temporary. Run this cell before disconnecting.

In [ ]:
# Copy only checkpoints and results back, preserving the scene layout.
relative_base = ('scenes/' + SCENE + '/') if SCENE else ''
drive_base = Path(DRIVE_PROJECT) / relative_base
local_base = Path(WORK) / relative_base
!mkdir -p "{drive_base}/runs" "{drive_base}/results"
!rsync -a "{local_base}/runs/{SRM_RUN}" "{drive_base}/runs/"
!rsync -a "{local_base}/runs/{FLOW_RUN}" "{drive_base}/runs/"
!rsync -a "{local_base}/results/{SRM_RUN}" "{drive_base}/results/"
!rsync -a "{local_base}/results/{FLOW_RUN}" "{drive_base}/results/"
print('Saved to', drive_base)

## Interpreting blur

Before attributing blur to the optimizer, check three causes:

- **BRDF blur:** the SRM is illumination convolved with surface roughness; this is physically expected.
- **Pose/normal error:** inaccurate geometry maps one environment feature into neighboring panorama directions.
- **Coverage:** unobserved directions cannot be recovered from the frames. A generative prior can make them plausible, but not ground-truth.

A genuinely sharp environment map requires either a mirror-like surface, explicit BRDF estimation/deconvolution, or a pretrained environment-map prior trained on many scenes.